In [1]:
import sys
import django
import os
file_dir = "/Users/mirbilal/Desktop/minsir/"
if file_dir not in sys.path:
    sys.path.insert(0, file_dir)

os.environ["DJANGO_SETTINGS_MODULE"] = "minsirx.settings"
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true" 
django.setup()

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from apps.email_manager.service_layer.email_linker import ConversationReader
from datetime import datetime
from apps.email_manager.models.email import EmailAttachment

In [2]:



convo_reader = ConversationReader(
    start_date=datetime(year=2024, month=7, day=10),
    end_date=datetime(year=2024, month=7, day=11),
)
convo_reader.extract_conversation()
convos = []
for a_convo_str, a_convo_data in convo_reader.conversations.items():
    # if 'Surveyor Appointed for Ticket No:' in a_convo_str:
        (full_convo, attachments_file_paths, all_files) = a_convo_data.get_full_convo()
        convos.append({
                "convo": full_convo,
                "attachment_file_paths": attachments_file_paths
        })

/Users/mirbilal/Desktop/minsir/minsirenv/lib/python3.12/site-packages/django/db/models/fields/__init__.py:1659: RuntimeWarning: DateTimeField EmailData.date received a naive datetime (2024-07-10 00:00:00) while time zone support is active.
  warnings.warn(
/Users/mirbilal/Desktop/minsir/minsirenv/lib/python3.12/site-packages/django/db/models/fields/__init__.py:1659: RuntimeWarning: DateTimeField EmailData.date received a naive datetime (2024-07-11 00:00:00) while time zone support is active.
  warnings.warn(


In [28]:
convos[0]

{'convo': [{'from': 'zoha.naeem@medtronic.com',
   'to': ['feroze.vakil@adamjeeinsurance.com',
    'mir.babarali@adamjeeinsurance.com'],
   'date': 'July 10, 2024, 4:22 AM',
   'subject': 'RE: APV & NV | 1994649  |1060437712 | 02-Jul-24',
   'body': 'Hi Mr. Babar/ Feroze,\n\xa0\nKind reminder\n\xa0\nBest Regards,\n\xa0\nZoha Naeem\n \n(MBA, CSCM)\nSr. Supply Planner | Customer Care and Supply Chain\n\xa0\n'}],
 'attachment_file_paths': ['/Users/mirbilal/Desktop/minsir/media/email_attachments/2024-6-26-0001994649_001_1jvy4s1.pdf']}

In [3]:
documents = SimpleDirectoryReader("/Users/mirbilal/Desktop/minsir/media/email_attachments/").load_data()
# also store all the email convos somewhere
index = VectorStoreIndex.from_documents(documents)

Multiple definitions in dictionary at byte 0x9d0a0 for key /Info
Multiple definitions in dictionary at byte 0x9d0ac for key /Info
Multiple definitions in dictionary at byte 0x9d0b8 for key /Info
Multiple definitions in dictionary at byte 0x9d0a0 for key /Info
Multiple definitions in dictionary at byte 0x9d0ac for key /Info
Multiple definitions in dictionary at byte 0x9d0b8 for key /Info


In [6]:
query_engine = index.as_query_engine()
resp = query_engine.query("were there any policies for meditronics?")

In [7]:
type(resp)

llama_index.core.base.response.schema.Response

In [8]:
resp

Response(response='Medtronic manufactures medical devices in accordance with several quality standards and regulations, such as Medtronic Quality Systems, USFDA Quality System Regulation (21 CFR 820), EN ISO 13485, and various ISO standards related to sterilization and quality management.', source_nodes=[NodeWithScore(node=TextNode(id_='34b2ba7f-86f0-43d9-ac8e-aa04bdbe5680', embedding=None, metadata={'page_label': '7', 'file_name': '2024-6-26-0001994649_001_JV3m8Ms.pdf', 'file_path': '/Users/mirbilal/Desktop/minsir/media/email_attachments/2024-6-26-0001994649_001_JV3m8Ms.pdf', 'file_type': 'application/pdf', 'file_size': 1317504, 'creation_date': '2024-07-10', 'last_modified_date': '2024-07-10'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.S

In [9]:
def format_llama_response(resp):
    # Extract the main response text
    print("Main Response:")
    print(f"{resp.response}\n")
    
    print("Source Nodes:")
    
    # Loop through the source nodes and display the metadata and content
    for i, node_with_score in enumerate(resp.source_nodes, 1):
        node = node_with_score.node
        metadata = node.metadata
        print(f"--- Source {i} ---")
        print(f"Page Label: {metadata.get('page_label')}")
        print(f"File Name: {metadata.get('file_name')}")
        print(f"File Path: {metadata.get('file_path')}")
        print(f"File Type: {metadata.get('file_type')}")
        print(f"File Size: {metadata.get('file_size')} bytes")
        print(f"Creation Date: {metadata.get('creation_date')}")
        print(f"Last Modified Date: {metadata.get('last_modified_date')}")
        print(f"Content (Excerpt):\n{node.text[:500]}...\n")  # Print a short excerpt of the content
        print(f"Score: {node_with_score.score}")
        print("\n")

# Example usage with your response object
format_llama_response(resp)

Main Response:
Medtronic manufactures medical devices in accordance with several quality standards and regulations, such as Medtronic Quality Systems, USFDA Quality System Regulation (21 CFR 820), EN ISO 13485, and various ISO standards related to sterilization and quality management.

Source Nodes:
--- Source 1 ---
Page Label: 7
File Name: 2024-6-26-0001994649_001_JV3m8Ms.pdf
File Path: /Users/mirbilal/Desktop/minsir/media/email_attachments/2024-6-26-0001994649_001_JV3m8Ms.pdf
File Type: application/pdf
File Size: 1317504 bytes
Creation Date: 2024-07-10
Last Modified Date: 2024-07-10
Content (Excerpt):
CERTIFICATE of STERILIZATION 
AND 
CERTIFICATE of QUALITY AND CONFORMITY 
Medtronic manufactures medical devices in accordance with the following, as applicable: 
• Medtronic Quality Systems 
• USFDA Quality System Regulation (21 CFR 820) 
• EN ISO 13485 Medical devices: Quality management systems 
• ISO 10993-1 Biological evaluation of medical devices 
• ISO 10993-7 Biological evaluati